# 01 - Data Exploration: NYC Yellow Taxi (January 2024)

**Goal:** Understand the dataset to be used before building any engine: how big it is on disk vs in memory, which columns cost the most, and how dirty the data is. Findings in this step will be used for further optimization.

**Data:** NYC TLC Yellow Taxi Trip Records, January 2024 (Parquet, 2,964,624 rows).

**Source:** NYC Taxi & Limousine Commission, Trip Record Data page.

## Key findings
- 50.0 MB on disk becomes 535.9 MB in memory (**10.7x**).

- One text column (`store_and_fwd_flag`) uses ~27% of all memory;
  converting it to `category` shrinks it from 144.6 MB to 3.0 MB.
  
- Reading only 4 necessary columns requires 94.9 MB instead of 535.9 MB.

- About 3.56% of the total rows appear invalid, according to the three different validity rule:
  - (duration outside 1-180 min range): 1.25%
  - (distance <= 0): 2.04%
  - (fare <= 0): 1.29%.


## Why this matters for the project
The router cannot estimate memory from file size alone: it depends on
the columns read and their data types.

In [1]:
# downloading 1 month of NYC taxi data

!mkdir -p data/raw
!wget -nc -q -P data/raw https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
!ls -lh data/raw

total 48M
-rw-r--r-- 1 root root 48M Mar 21  2024 yellow_tripdata_2024-01.parquet


In [2]:
# measuring file size vs memory size

import pandas as pd
from pathlib import Path

path = Path("data/raw/yellow_tripdata_2024-01.parquet")
disk_mb = path.stat().st_size / 1e6

df = pd.read_parquet(path)
mem_mb = df.memory_usage(deep=True).sum() / 1e6

print(f"Rows: {len(df):,}")
print(f"Size on disk: {disk_mb:.1f} MB \nSize in memory: {mem_mb:.1f} MB \nRatio: {mem_mb/disk_mb:.1f}x")

Rows: 2,964,624
Size on disk: 50.0 MB 
Size in memory: 535.9 MB 
Ratio: 10.7x


In [3]:
# first look at the data

df.info()
df.isna().sum()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

,count,mean,min,25%,50%,75%,max,std
VendorID,2964624.0,1.754204,1.0,2.0,2.0,2.0,6.0,0.43259
tpep_pickup_datetime,2964624,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15,NaN
tpep_dropoff_datetime,2964624,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52,NaN
passenger_count,2824462.0,1.339281,0.0,1.0,1.0,1.0,9.0,0.850282
trip_distance,2964624.0,3.652169,0.0,1.0,1.68,3.11,312722.3,225.462572
RatecodeID,2824462.0,2.069359,1.0,1.0,1.0,1.0,99.0,9.823219
PULocationID,2964624.0,166.017884,1.0,132.0,162.0,234.0,265.0,63.623914
DOLocationID,2964624.0,165.116712,1.0,114.0,162.0,234.0,265.0,69.31535
payment_type,2964624.0,1.161271,0.0,1.0,1.0,1.0,4.0,0.580869
fare_amount,2964624.0,18.175062,-899.0,8.6,12.8,20.5,5000.0,18.949548


In [4]:
# reading only the columns we need

cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount"]
df_small = pd.read_parquet(path, columns=cols)
print(f"Size in memory: {df_small.memory_usage(deep=True).sum()/ 1e6} MB")


Size in memory: 94.8681 MB


In [5]:
# memory by column

mem = (df.memory_usage(deep=True).sort_values(ascending=False) / 1e6)
print("The 5 columns that take up the most space:")
print(mem.head(5))
print()

The 5 columns that take up the most space:
store_and_fwd_flag       144.586988
tpep_dropoff_datetime     23.716992
passenger_count           23.716992
tpep_pickup_datetime      23.716992
improvement_surcharge     23.716992
dtype: float64



In [6]:
# dtype optimization example

b1 = df["store_and_fwd_flag"].memory_usage(deep=True) / 1e6
a1 = df["store_and_fwd_flag"].astype("category").memory_usage(deep=True) / 1e6
print(f"Data size in memory after changing data type for store_and_fwd_flag column: \nBefore: {b1:.2f} MB (Datatype: object) \nAfter: {a1:.2f} MB (Datatype: category)")


b2 = df["passenger_count"].memory_usage(deep=True) / 1e6
a2 = df["passenger_count"].astype("Int8").memory_usage(deep=True) / 1e6
print(f"\nData size in memory after changing data type for passenger_count column: \nBefore: {b2:.2f} MB (Datatype: float64) \nAfter: {a2:.2f} MB (Datatype: Int8)")


Data size in memory after changing data type for store_and_fwd_flag column: 
Before: 144.59 MB (Datatype: object) 
After: 2.96 MB (Datatype: category)

Data size in memory after changing data type for passenger_count column: 
Before: 23.72 MB (Datatype: float64) 
After: 5.93 MB (Datatype: Int8)


In [7]:
# data quality check
# counting rows that break some validity rules

trip_min = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

bad_duration = ((trip_min < 1) | (trip_min > 180))
bad_distance = (df["trip_distance"] <= 0)
bad_fare = (df["fare_amount"] <= 0)
any_invalid = bad_duration | bad_distance | bad_fare

rules = {
    "Number of rows where duration is outside of 1-180 min range": bad_duration,
    "Number of rows where 'distance <= 0'": bad_distance,
    "Number of rows where 'fare <= 0'": bad_fare,
    "ANY rule (combined)": any_invalid,
}

for rule, mask in rules.items():
  print(f"{rule}: {mask.sum():,}\nWhich is {(mask.mean()):.2%} of total number of rows.\n")



Number of rows where duration is outside of 1-180 min range: 37,104
Which is 1.25% of total number of rows.

Number of rows where 'distance <= 0': 60,371
Which is 2.04% of total number of rows.

Number of rows where 'fare <= 0': 38,341
Which is 1.29% of total number of rows.

ANY rule (combined): 105,602
Which is 3.56% of total number of rows.

